# Classification Notebook - Char 1 Only

This notebook contains machine learning classification algorithms for single character (char 1) epitope prediction.

## Features:
- GPU/CPU training option for all ML algorithms
- Support for multiple propensity scales
- Model saving and evaluation metrics

In [1]:
# Configuration
import os

# Set device preference: 'gpu' or 'cpu'
DEVICE = 'cpu'  # Change to 'gpu' if GPU is available

# Model directory
model_dir = 'model/'
os.makedirs(model_dir, exist_ok=True)

# Check GPU availability
import tensorflow as tf

if DEVICE == 'gpu':
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print(f"GPU available: {len(gpus)} GPU(s)")
        for gpu in gpus:
            print(f"  - {gpu}")
    else:
        print("GPU requested but not available. Falling back to CPU.")
        DEVICE = 'cpu'
else:
    print("Using CPU for training")

2025-12-26 06:03:44.294186: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766729024.552485      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766729024.622659      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Using CPU for training


# Machine Learning Classification Algorithms - Char 1

In [2]:
import pickle

# Save model to file
def save_pkl(model, name):
    with open(f'{model_dir}{name}.pkl', 'wb') as f:
        pickle.dump(model, f)

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.metrics import roc_auc_score

def dl(df, X, y, data_test):
    """
    Deep Learning model with GPU/CPU support
    """
    # Configure device
    if DEVICE == 'gpu':
        with tf.device('/GPU:0'):
            return _train_dl(df, X, y, data_test)
    else:
        with tf.device('/CPU:0'):
            return _train_dl(df, X, y, data_test)

def _train_dl(df, X, y, data_test):
    # Drop missing values in the "Position" column
    df = df.dropna(subset=["Position"])

    # Preprocessing data
    X_data = df[X]
    y_data = df[y].values

    # Encode labels
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y_data)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_data, y_encoded, test_size=data_test, random_state=42)

    # Initialize model
    model = Sequential()
    model.add(Dense(16, activation='relu', input_dim=X_train.shape[1]))
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))

    # Compile model
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

    # Train model
    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)

    # Predict
    y_pred_proba = model.predict(X_test, verbose=0)

    # Calculate AUC
    auc = roc_auc_score(y_test, y_pred_proba)

    # Evaluate
    _, accuracy = model.evaluate(X_test, y_test, verbose=0)

    # Save model
    model.save(model_dir+'deep_learning_model.h5')

    return accuracy, auc

In [4]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score

def nn(df, X, y, data_test):
    """
    Neural Network (MLP) model
    Note: sklearn MLPClassifier uses CPU by default
    """
    # Drop missing values
    df = df.dropna(subset=["Position"])

    # Preprocessing
    X_data = df[X]
    y_data = df[y].values

    # Encode labels
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y_data)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_data, y_encoded, test_size=data_test, random_state=42)

    # Initialize model
    model = MLPClassifier()

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # Calculate AUC
    auc = roc_auc_score(y_test, y_pred_proba)

    # Save model
    save_pkl(model, 'nnMLP')

    # Calculate accuracy
    accuracy = (y_pred == y_test).mean()

    return accuracy, auc


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def rf(df, X, y, data_test):
    """
    Random Forest Classifier
    Note: sklearn RandomForest uses CPU by default
    """
    # Drop missing values
    df = df.dropna(subset=["Position"])

    # Preprocessing
    X_data = df[X]
    y_data = df[y].values

    # Encode labels
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y_data)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_data, y_encoded, test_size=data_test, random_state=42)

    # Initialize model
    model = RandomForestClassifier()

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # Calculate AUC
    auc = roc_auc_score(y_test, y_pred_proba)

    # Save model
    save_pkl(model, 'RandomForest')

    # Calculate accuracy
    accuracy = (y_pred == y_test).mean()

    return accuracy, auc


In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

def dt(df, X, y, data_test):
    """
    Decision Tree Classifier
    """
    # Drop missing values
    df = df.dropna(subset=["Position"])

    # Preprocessing
    X_data = df[X]
    y_data = df[y].values

    # Encode labels
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y_data)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_data, y_encoded, test_size=data_test, random_state=42)

    # Initialize model
    model = DecisionTreeClassifier()

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # Calculate AUC
    auc = roc_auc_score(y_test, y_pred_proba)

    # Save model
    save_pkl(model, 'DecisionTree')

    # Calculate accuracy
    accuracy = (y_pred == y_test).mean()

    return accuracy, auc


In [7]:
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score

def svm(df, X, y, data_test):
    """
    Support Vector Machine Classifier
    """
    # Drop missing values
    df = df.dropna(subset=["Position"])

    # Preprocessing
    X_data = df[X]
    y_data = df[y].values

    # Encode labels
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y_data)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_data, y_encoded, test_size=data_test, random_state=42)

    # Initialize model
    model = SVC(probability=True)

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # Calculate AUC
    auc = roc_auc_score(y_test, y_pred_proba)

    # Save model
    save_pkl(model, 'SVM')

    # Calculate accuracy
    accuracy = (y_pred == y_test).mean()

    return accuracy, auc


In [8]:
from hmmlearn import hmm
from sklearn.metrics import roc_auc_score

def HMM(df, X, y, data_test):
    """
    Hidden Markov Model Classifier
    """
    # Drop missing values
    df = df.dropna(subset=["Position"])

    # Preprocessing
    X_data = df[X]
    y_data = df[y].values

    # Encode labels
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y_data)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_data, y_encoded, test_size=data_test, random_state=42)

    # Initialize model
    model = hmm.GaussianHMM(n_components=2)

    # Train model
    model.fit(X_train)

    # Predict
    y_pred = model.predict(X_test)
    
    # Get probabilities
    try:
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    except:
        # Fallback if predict_proba not available
        y_pred_proba = y_pred

    # Calculate AUC
    auc = roc_auc_score(y_test, y_pred_proba)

    # Save model
    save_pkl(model, 'hmmmodel')

    # Calculate accuracy
    accuracy = (y_pred == y_test).mean()

    return accuracy, auc


ModuleNotFoundError: No module named 'hmmlearn'

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from hmmlearn import hmm
from sklearn.metrics import roc_auc_score

def HMM(df, X, y,data_test):
    # Drop missing values in the "Position" column
    df = df.dropna(subset=["Position"])

    # Preprocessing data
    X = df[X]
    y = df[y].values

    # Mengubah label menjadi bilangan bulat
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # Membagi data menjadi data latih dan data uji
    X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=data_test, random_state=42)

    # Menginisialisasi model HMM dengan Gaussian emissions
    model = hmm.GaussianHMM(n_components=2)

    # Melatih model HMM dengan data latih
    model.fit(X_train)

    # Memprediksi label pada data uji
    y_pred = model.predict(X_test)

    # Mengubah label hasil prediksi menjadi label awal
    y_pred_label = label_encoder.inverse_transform(y_pred)

    # Menghitung probabilitas prediksi pada data uji
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # Menghitung nilai AUC (Area Under Curve)
    auc = roc_auc_score(y_test, y_pred_proba)

    save_pkl(model,'hmmmodel')

    # Menghitung akurasi
    accuracy = (y_pred_label == label_encoder.inverse_transform(y_test)).mean()

    # Mengembalikan akurasi dan nilai ROC
    return accuracy, auc


# Penerapan

In [ ]:
import pandas as pd

In [ ]:
df1 = pd.read_csv('dataset/dataset_type_2_vers2_hidropobicity.csv')
df2 = pd.read_csv('dataset/dataset_type_2_char2_vers2.csv')

In [ ]:
42/3

# Results Summary


In [ ]:
# Display summary statistics
print("Summary Statistics:")
print(dtest.groupby(['algoritm', 'test_size'])[['accuracy', 'auc']].mean())
print("\nBest Results:")
print(dtest.loc[dtest['auc'].idxmax()])


In [ ]:
dtest = pd.DataFrame()
mod = ['hmm','nn','dl','dt','rf','svm']
#prop_scale = ['hoop_woods','emini','parker','levitt']
prop_scale = ['Kyte-Doolittle', 'Hopp-Woods', 'Cornette', 'Eisenberg', 'Rose', 'Janin', 'Engelman GES']
dt_test = [0.1,0.2,0.3]
mod4x = mod*len(prop_scale)
prop_scale6x = prop_scale*len(mod)
dtest['algoritm'] = mod4x
dtest['prop_scale'] = prop_scale6x
dtest['accuracy'] = 0.0
dtest['auc'] = 0.0
dtest['n_amino_acids'] = 1
dtest['test_size'] = 0.0
dtest = pd.concat([dtest,dtest,dtest])
dtest = dtest.reset_index(drop=True)
#dtest = dtest.sample(frac=1).reset_index(drop=True)
for i in mod:
    dtest.loc[dtest['algoritm'] == i, 'test_size'] = int(len(dtest[dtest['algoritm'] == i])/len(dt_test))*dt_test

In [ ]:
len(dtest)

# Penerapan Char 1

In [ ]:
dtest = pd.read_csv('result_of_classification.csv')

In [ ]:
begin = dtest[dtest['accuracy'] == 0.00]

In [ ]:
begin.index[0]

In [ ]:
y = 'label'
for i in range(begin.index[0],len(dtest)):
    X = ['Position z-score']

    if(dtest['algoritm'][i] == 'hmm'):
        X.append(dtest['prop_scale'][i])
        acc, auc = HMM(df1,X, y,dtest['test_size'][i])
    
    if(dtest['algoritm'][i] == 'nn'):
        X.append(dtest['prop_scale'][i])
        acc, auc = nn(df1,X, y,dtest['test_size'][i])
    
    if(dtest['algoritm'][i] == 'dl'):
        X.append(dtest['prop_scale'][i])
        acc, auc = dl(df1,X, y,dtest['test_size'][i])
    
    if(dtest['algoritm'][i] == 'dt'):
        X.append(dtest['prop_scale'][i])
        acc, auc = dt(df1,X, y,dtest['test_size'][i])
    
    if(dtest['algoritm'][i] == 'rf'):
        X.append(dtest['prop_scale'][i])
        acc, auc = rf(df1,X, y,dtest['test_size'][i])

    if(dtest['algoritm'][i] == 'svm'):
        X.append(dtest['prop_scale'][i])
        acc, auc = svm(df1,X, y,dtest['test_size'][i])

    dtest['accuracy'][i] = acc
    dtest['auc'][i] = auc

    dtest.to_csv('result_of_classification.csv',index=False)

# Penerapan Char 2

In [ ]:
df2

In [ ]:
X = ['Position', 'hoop_woods 1', 'hoop_woods 2']
y = 'label'
acc, auc = nn(df2,X, y)
print(acc)

In [ ]:
mod = ['hmm','nn','dl','dt','rf','svm']
prop_scale = ['hoop_woods','emini','parker','levitt']
mod4x = mod*len(prop_scale)
prop_scale6x = prop_scale*len(mod)
dtest['algoritm'] = mod4x
dtest['prop_scale'] = prop_scale6x
dtest['accuracy'] = 0.0
dtest['auc'] = 0.0
dtest['n_amino_acids'] = 2

In [ ]:
dtest

In [ ]:
y = 'label'
for i in range(len(dtest)):
    X = ['Position']

    if(dtest['algoritm'][i] == 'hmm'):
        X.append(dtest['prop_scale'][i]+" 1")
        X.append(dtest['prop_scale'][i]+" 2")
        acc, auc = HMM(df2,X, y)
    
    if(dtest['algoritm'][i] == 'nn'):
        X.append(dtest['prop_scale'][i]+" 1")
        X.append(dtest['prop_scale'][i]+" 2")
        acc, auc = nn(df2,X, y)
    
    if(dtest['algoritm'][i] == 'dl'):
        X.append(dtest['prop_scale'][i]+" 1")
        X.append(dtest['prop_scale'][i]+" 2")
        acc, auc = dl(df2,X, y)
    
    if(dtest['algoritm'][i] == 'dt'):
        X.append(dtest['prop_scale'][i]+" 1")
        X.append(dtest['prop_scale'][i]+" 2")
        acc, auc = dt(df2,X, y)
    
    if(dtest['algoritm'][i] == 'rf'):
        X.append(dtest['prop_scale'][i]+" 1")
        X.append(dtest['prop_scale'][i]+" 2")
        acc, auc = rf(df2,X, y)

    if(dtest['algoritm'][i] == 'svm'):
        X.append(dtest['prop_scale'][i]+" 1")
        X.append(dtest['prop_scale'][i]+" 2")
        acc, auc = svm(df2,X, y)

    dtest['accuracy'][i] = acc
    dtest['auc'][i] = auc

    dtest.to_csv('result_of_classification_char2.csv',index=False)

In [ ]:
# dijumlahkan propensity scale nya
y = 'label'
for i in range(len(dtest)):
    X = ['Position']

    if(dtest['algoritm'][i] == 'hmm'):
        X.append(dtest['prop_scale'][i])
        acc, auc = HMM(df2,X, y)
    
    if(dtest['algoritm'][i] == 'nn'):
        X.append(dtest['prop_scale'][i])
        acc, auc = nn(df2,X, y)
    
    if(dtest['algoritm'][i] == 'dl'):
        X.append(dtest['prop_scale'][i])
        acc, auc = dl(df2,X, y)
    
    if(dtest['algoritm'][i] == 'dt'):
        X.append(dtest['prop_scale'][i])
        acc, auc = dt(df2,X, y)
    
    if(dtest['algoritm'][i] == 'rf'):
        X.append(dtest['prop_scale'][i])
        acc, auc = rf(df2,X, y)

    if(dtest['algoritm'][i] == 'svm'):
        X.append(dtest['prop_scale'][i])
        acc, auc = svm(df2,X, y)

    dtest['accuracy'][i] = acc
    dtest['auc'][i] = auc

    dtest.to_csv('result_of_classification_char2_vers2.csv',index=False)